In [1]:
import json
from pathlib import Path

%load_ext autoreload
%autoreload 2

from utgen.raggraph.walker import build_graph_from_directory
from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew
from utgen.validation import validate_individual_test, save_and_clean_tests

In [5]:
g = build_graph_from_directory(code_path="../src", save_graph=False)

In [6]:
tests_results: dict[str, dict[str, dict[str, str]]] = {}

# TODO: afegir guardrails que falten
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

# Per a cada node del graf, obtenim el seu context i creem el test
for node_id, data in list(g.nodes(data=True))[20:35]:
    if data["type"] in ["function"]:
    # if data["type"] in ["function", "method"]:
        print(f"Generating tests for node: {node_id}")
        # Get context
        context = get_node_context(g=g, node_id=node_id)

        # Generate tests
        inputs = {"graph_context": context}
        response = test_generator.crew().kickoff(inputs=inputs)

        # Convert string to dictionary
        response_dict = json.loads(response.raw)

        # Store results
        tests_results[node_id] = response_dict['tests']

Generating tests for node: utgen/raggraph/utils.py::function::get_node_context
Generating tests for node: utgen/raggraph/utils.py::function::get_source_segment
Generating tests for node: utgen/raggraph/utils.py::function::canonical_id
Generating tests for node: utgen/raggraph/utils.py::function::normalize_signature



2026-03-16 17:24:18.959 | WARNING  | utgen.test_generation_crew.guardrails:validate_tests_schema:94 - Guardrail `validate_tests_schema` triggered: invalid test entries.


Generating tests for node: utgen/raggraph/walker.py::function::iter_python_files



2026-03-16 17:25:07.031 | WARNING  | utgen.test_generation_crew.guardrails:validate_tests_schema:53 - Guardrail `validate_tests_schema` triggered: invalid JSON format


Generating tests for node: utgen/raggraph/walker.py::function::build_graph_from_directory


In [14]:
counter = 1

for node_id, tests in tests_results.items():
    # Validem el test generat
    print(f"\nValidant tests per `{node_id}`...")
    base_import = 'from ' + node_id.split("::")[0][:-3].replace("/", ".") + ' import ' + node_id.split("::")[-1].split(".")[0]
    tests_que_han_passat = []

    for k2, v2 in tests.items():
        name, imp, code = k2, v2['imports'], v2['code']
        # Afegim el import base per si no està present
        if base_import not in imp:
            print(f"⚠️ Import base `{base_import}` no present en el test `{name}`, afegint-lo...")
            imp.append(base_import)
        imp = "\n".join(imp)
        if validate_individual_test(import_code=imp, test_code=code):
            print(f"✅ Test `{name}` acceptat")
            tests_que_han_passat.append((imp, code))
        else:
            print(f"❌ Test `{name}` rebutjat")
        print(imp)
        print(code)

    # Guardem i deixem el fitxer perfecte
    p = Path(g.nodes[node_id]["file"])
    new_filename = f"test_{p.stem}_{counter}{p.suffix}"
    final_path = (p.parent / new_filename).as_posix()

    print("📄", str(final_path))
    save_and_clean_tests(valid_tests=tests_que_han_passat, destination=f"../tests/{final_path}")
    counter += 1

# TODO: run pytest coverage


Validant tests per `utgen/raggraph/utils.py::function::get_node_context`...
✅ Test `test_get_node_context_with_basic_graph` acceptat
import pytest
from unittest.mock import Mock
import networkx as nx
from utgen.raggraph.utils import get_node_context
def test_get_node_context_with_basic_graph():
    """
    Test get_node_context with a basic graph containing a single node.
    """
    # Arrange
    g = nx.DiGraph()
    
    # Create a mock node with basic attributes
    node_id = "test.py::function::test_function"
    g.add_node(node_id, 
              source="def test_function():",
              type="function",
              signature="def test_function()",
              docstring="Test function docstring")
    
    # Act
    result = get_node_context(g, node_id)
    
    # Assert
    assert "### NODE INFO ###" in result
    assert f"file::type::name -> {node_id}" in result
    assert "source:" in result
    assert "def test_function():" in result
    assert "### OUTGOING EDGES ###" 

In [ ]:
from pathlib import Path

target_dir = "../tests"

# 1. Find all files ending in _[digit].py (e.g., test_utils_1.py)
for file_path in Path(target_dir).rglob("*_[0-9].py"):
    
    # 2. Define the new name (e.g., test_utils.py)
    unified_name = f"{file_path.stem[:-2]}.py"
    unified_path = file_path.parent / unified_name
    
    # 3. Read the code from the numbered file
    code = file_path.read_text(encoding="utf-8")
    
    # 4. Open the unified file in "append" mode ('a')
    with open(unified_path, "a", encoding="utf-8") as f:
        f.write(code)
    
    # 5. Delete the old numbered file
    file_path.unlink()

print("Merging complete.")

Merging complete.


In [5]:
import subprocess
from pathlib import Path

destination = "../tests"

def hoist_imports(target_path):
    """Physically moves all 'import' and 'from' lines to the top of the file."""
    for file_path in Path(target_path).rglob("*.py"):
        lines = file_path.read_text(encoding="utf-8").splitlines()
        
        imports = []
        body = []
        
        for line in lines:
            # Check if the line is an import (ignoring leading whitespace)
            stripped = line.strip()
            if stripped.startswith(("import ", "from ")):
                imports.append(line)
            else:
                body.append(line)
        
        # Rewrite the file: Imports first, then the rest
        file_path.write_text("\n".join(imports + [""] + body), encoding="utf-8")

# 1. Manually move imports to the top so Ruff can see them as one block
print(f"Hoisting imports in {destination}...")
hoist_imports(destination)

# 2. Use Ruff to organize and clean
print(f"Cleaning {destination} with Ruff...")

# Sort, de-duplicate, and fix "imports not at top" (E402)
# Adding E402 tells Ruff it's okay to fix the position of imports
subprocess.run(["ruff", "check", "--select", "I,E402,F401", "--fix", destination], capture_output=True)

# Final linting and formatting
subprocess.run(["ruff", "check", "--fix", destination], capture_output=True)
subprocess.run(["ruff", "format", destination], capture_output=True)

print(f"✅ Process finished. Files in {destination} are unified and cleaned.")

Hoisting imports in ../tests...
Cleaning ../tests with Ruff...
✅ Process finished. Files in ../tests are unified and cleaned.
